# Calibrate Using Different Methods

In [ ]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

import fates_calibration_library.emulator_functions as em
import fates_calibration_library.calibration as cal
import fates_calibration_library.parameter_generation as param
import fates_calibration_library.utils as utils

import importlib

## Set Up
Load files, set up ensemble information

In [ ]:
top_dir = '/glade/work/afoster/FATES_calibration'
param_dir = os.path.join(top_dir, 'parameter_files')
emulator_dir = os.path.join(top_dir, 'emulators')

config_file = os.path.join(top_dir, 'emulator_configs', 'codom_pft.yaml')
calib_var_file = os.path.join(top_dir, 'emulator_configs', 'calibration_vars.yaml')
param_update_file = os.path.join(top_dir, 'emulator_configs', 'param_min_max.yaml')
pft_id_config = os.path.join(top_dir, 'fates_calibration_library/configs/fates_pft_ids.yaml')
obs_config_file = os.path.join(top_dir, 'fates_calibration_library/configs/ilamb_conversion.yaml')

param_file_name = 'fates_params_default_sci.1.85.1_api.40.0.0_crops.nc'
param_list_name = "param_list_sci.1.85.1_api.40.0.0.xls"
default_param = xr.open_dataset(os.path.join(param_dir, param_file_name))
param_list_file = os.path.join(param_dir, param_list_name)
param_dat = param.get_param_dictionary(param_list_file)

In [ ]:
ensemble_config = utils.get_config_file(config_file)
calib_vars_config = utils.get_config_file(calib_var_file)
pft_ids = utils.get_config_file(pft_id_config)
obs_config = utils.get_config_file(obs_config_file)

In [ ]:
# Latin Hypercube information
param_names, default_norm = em.load_lhc_metadata(ensemble_config)
num_params = len(param_names)

## Chose PFT

In [ ]:
pft = 14

pft_name, pft_id = param.get_pft_info(pft, ensemble_config['default_param'],
                                      pft_ids)
calibration_vars = calib_vars_config[pft_id]

## Load required objects

In [ ]:
# emulators and targets
emulators, targets, sds, sens_pft = cal.load_emulator_and_obs_data(
    ensemble_config, pft_name, pft_id, emulator_dir, calibration_vars,
    obs_config
)

In [ ]:
# default parameters
params_default = cal.get_default_pft_values(default_norm, pft)

In [ ]:
# indices for to fix and optimize
fixed_indices, optimize_indices, num_optimize = cal.get_params_to_optimize(sens_pft,
                                                                          param_names, 
                                                                          num_params,
                                                                           sobol_threshold=0.1)

In [ ]:
# optimization config
opt_config = {
        'maxiter': 5000,
        'epsilon': 0.5,
        'lambda_penalty': None,
        'barrier_strength': 0,
        'loss_fn': cal.squared_z_loss,
        'default_penalty_fn': cal.default_penalty_l1,
        'barrier_penalty_fn': cal.barrier_penalty,
        'tol': 1e-3
}

## Calibration

This method uses scipy optimize

In [ ]:
# all_results = cal.run_batch_optimization(emulators, targets, sds, fixed_indices,
#                                         params_default, num_optimize,
#                                         param_names, optimize_indices,
#                                         opt_config, param_update_config,
#                                         num_batch=100)

## History Matching Approach

We are just going to try to emulate a bunch

In [ ]:
if pft == 14:
    is_ch4 = True
else:
    is_ch4 = False

In [ ]:
n_bootstraps = 50
results_list = []
for i in range(n_bootstraps):
    result_row = cal.run_history_matching(emulators, calib_vars_config[pft_id],
                                          targets, sds, param_names,
                                          implaus_tol=1.0, is_ch4=is_ch4)
    if result_row is not None:
        result_row['run_id'] = i
        results_list.append(result_row)
final_results_df = pd.concat(results_list, axis=0, ignore_index=True)